# 02 — TMDB Matching & Enrichment

Resolves each film to a TMDB record and pulls the metadata that becomes Block F.

**Input:** `data/interim/viewings.csv` (from `01`)
**Output:** `data/interim/films_enriched.csv` — one row per film with TMDB metadata

Two stages. **Matching** decides *which* TMDB record corresponds to a Letterboxd
entry, which is harder than it sounds — TMDB's relevance ranking is unreliable and
title collisions are common. **Enrichment** then fetches facts about the matched
film.

Both stages cache raw API responses to disk, so reruns cost no API calls and the
parsing/scoring logic can be changed freely.


## 1. Setup

`include_adult=True` matters: TMDB excludes adult-flagged titles from search by
default, which silently hides legitimate arthouse films. *Antiporno* (Sion Sono,
2016) was invisible without it — only a making-of documentary came back.


In [1]:
import os
import requests
import pandas as pd
import re
import unicodedata
import json
import time

from dotenv import load_dotenv
from pathlib import Path
from difflib import SequenceMatcher


load_dotenv()
TMDB_TOKEN = os.getenv("TMDB_TOKEN")

HEADERS = {"Authorization": f"Bearer {TMDB_TOKEN}", "accept": "application/json"}

def search_film(title, year=None):
    params = {"query": title, "include_adult": True}
    if year:
        params["year"] = year
    r = requests.get("https://api.themoviedb.org/3/search/movie",
                     params=params, headers=HEADERS, timeout=15)
    r.raise_for_status()
    return r.json()["results"]

results = search_film("Dune", 2021)
print(f"{len(results)} results\n")
results[0]

20 results



{'adult': False,
 'backdrop_path': '/zRKQW58MBEY078AxkHxEJzUskCl.jpg',
 'genre_ids': [878, 12],
 'id': 438631,
 'title': 'Dune',
 'original_language': 'en',
 'original_title': 'Dune',
 'overview': "Paul Atreides, a brilliant and gifted young man born into a great destiny beyond his understanding, must travel to the most dangerous planet in the universe to ensure the future of his family and his people. As malevolent forces explode into conflict over the planet's exclusive supply of the most precious resource in existence - a commodity capable of unlocking humanity's greatest potential - only those who can conquer their fear will survive.",
 'popularity': 49.2188,
 'poster_path': '/v1tRXZ4JtD2Iv6fjkPvT4GiwslV.jpg',
 'release_date': '2021-09-15',
 'softcore': False,
 'video': False,
 'vote_average': 7.785,
 'vote_count': 15557}

## 2. Load films to match

Deduplicated to unique films: rewatches are separate viewing events but the same
film, so matching per-viewing would waste API calls and produce duplicate records.


In [2]:

viewings = pd.read_csv("data/interim/viewings.csv")

films = (viewings
         .drop_duplicates("film_key")
         [["film_key", "film_title", "film_year"]]
         .reset_index(drop=True))

print(f"{len(viewings)} viewings -> {len(films)} unique films to match")
films.head()

1194 viewings -> 1176 unique films to match


,film_key,film_title,film_year
0,Don't Look Up (2021),Don't Look Up,2021
1,Tenet (2020),Tenet,2020
2,Memories of Murder (2003),Memories of Murder,2003
3,Dead Poets Society (1989),Dead Poets Society,1989
4,The Mitchells vs. the Machines (2021),The Mitchells vs. the Machines,2021


## 3. Title normalisation

Both titles are flattened before comparison so trivial differences don't count as
mismatches: lowercase, accents stripped, punctuation replaced with spaces,
whitespace collapsed.

Accent stripping uses NFKD normalisation to split accented characters into base
letter plus combining mark, then drops the marks — so *Amélie* and *Amelie* compare
as identical, including when the source uses precomposed vs decomposed Unicode.

Punctuation becomes a space rather than being deleted, so `3-Iron` → `3 iron` rather
than `3iron`.


In [3]:
def normalise(title):
    """Flatten a title for comparison: lowercase, no accents, no punctuation."""
    if not isinstance(title, str):
        return ""
    title = unicodedata.normalize("NFKD", title)
    title = "".join(c for c in title if not unicodedata.combining(c))
    title = title.casefold()
    title = re.sub(r"[^\w\s]", " ", title)
    return re.sub(r"\s+", " ", title).strip()

for t in ["Amélie", "The Boy, the Mole, the Fox and the Horse", "WALL·E",
          "Am\u00e9lie", "Don't Look Up", "3-Iron"]:
    print(f"{t!r:45s} -> {normalise(t)!r}")

'Amélie'                                      -> 'amelie'
'The Boy, the Mole, the Fox and the Horse'    -> 'the boy the mole the fox and the horse'
'WALL·E'                                      -> 'wall e'
'Amélie'                                      -> 'amelie'
"Don't Look Up"                               -> 'don t look up'
'3-Iron'                                      -> '3 iron'


## 4. Scoring a candidate

Four probes against the live API established what this has to handle:

| Probe | Finding |
|---|---|
| `Dune` (2021) | 20 results even with the year filter — TMDB's `year` is a soft filter, so `results[0]` cannot be trusted |
| `Memories of Murder` (2003) | `title` is localised English, `original_title` is 살인의 추억 — must compare against **both** |
| `The Fall` (2019) | *Legends of the Fall* ranked **first**; two different 2019 films share the exact title |
| `The Big Shave` (1967) | Obscure 1967 short matched cleanly — TMDB's older coverage is better than expected |

Tiers rather than a composite score: easier to review, and easier to explain.
`exact` and `year_off` are auto-accepted, everything else is inspected.


In [4]:
def similarity(a, b):
    return SequenceMatcher(None, normalise(a), normalise(b)).ratio()

def score_candidate(candidate, title, year):
    """Classify one TMDB search result against the Letterboxd title and year."""
    sim = max(similarity(title, candidate.get("title") or ""),
              similarity(title, candidate.get("original_title") or ""))

    release = candidate.get("release_date") or ""
    cand_year = int(release[:4]) if release[:4].isdigit() else None
    gap = None if (year is None or cand_year is None) else abs(cand_year - year)

    if sim >= 0.95 and gap == 0:
        return "exact", sim
    if sim >= 0.95 and gap == 1:
        return "year_off", sim
    if sim >= 0.85 and (gap is None or gap <= 1):
        return "close", sim
    return "weak", sim

In [5]:
for r in search_film("The Fall", 2019)[:5]:
    tier, sim = score_candidate(r, "The Fall", 2019)
    print(f"{tier:10s} {sim:.3f}  {r['title']!r} ({(r.get('release_date') or '?')[:4]})")

weak       0.593  'Legends of the Fall' (1994)
exact      1.000  'The Fall' (2019)
exact      1.000  'The Fall' (2019)
weak       0.727  'The Hours Fall' (2018)
weak       0.640  'The Fall of Cabal' (2019)


## 5. Selecting the best candidate

Sorted on tier, then similarity, then vote count. The negatives flip sort direction —
lower tier index is better, but *higher* similarity and vote count are better.

**The demotion rule.** An early version failed on *Enemy* (2013): Villeneuve's film
is dated 2014 on TMDB so it scored `year_off`, while an unrelated Bengali film of the
same name and year scored `exact` and won on tier. Same failure hit *Under the Skin*,
*Talk to Me* and *Presence* — in each case a short or fragment sharing a title with a
feature.

So a candidate is demoted one tier when its vote count is under 5% of the
best-known genuine title match. This only fires when there is competition, leaving
genuinely obscure films untouched: *Beijing Watermelon* has 7 votes but no
better-known rival, so nothing demotes.


In [6]:
TIERS = ["exact", "year_off", "close", "weak"]
DEMOTE = {"exact": "close", "year_off": "close", "close": "weak", "weak": "weak"}
MIN_VOTE_SHARE = 0.05        # under 5% of the best-known rival's votes = implausible

def best_match(results, title, year):
    """Pick the strongest candidate, demoting implausibly obscure title collisions."""
    if not results:
        return None

    scored = []
    for c in results[:10]:
        tier, sim = score_candidate(c, title, year)
        scored.append({"tier": tier, "sim": sim,
                       "votes": c.get("vote_count") or 0, "raw": c})

    # among candidates whose title genuinely matches, how well-known is the best?
    plausible = [s["votes"] for s in scored if s["sim"] >= 0.85]
    max_votes = max(plausible) if plausible else 0

    for s in scored:
        if s["sim"] >= 0.85 and max_votes > 0 and s["votes"] < max_votes * MIN_VOTE_SHARE:
            s["tier"] = DEMOTE[s["tier"]]

    scored.sort(key=lambda s: (TIERS.index(s["tier"]), -s["sim"], -s["votes"]))
    best = scored[0]
    c = best["raw"]

    release = c.get("release_date") or ""
    return {
        "tmdb_id":       c.get("id"),
        "matched_title": c.get("title"),
        "matched_year":  int(release[:4]) if release[:4].isdigit() else None,
        "confidence":    best["tier"],
        "similarity":    round(best["sim"], 3),
        "vote_count":    c.get("vote_count"),
    }

In [7]:
best_match(search_film("The Fall", 2019), "The Fall", 2019)

{'tmdb_id': 643065,
 'matched_title': 'The Fall',
 'matched_year': 2019,
 'confidence': 'exact',
 'similarity': 1.0,
 'vote_count': 110}

## 6. Matching every film

The cache stores the **raw** search response, not the parsed match, so any change to
`score_candidate` or `best_match` can be re-run at zero API cost. (An earlier version
cached the parsed result and had to re-fetch all 1,174 films when the scoring
changed.)

The year-filtered search is retried without the year when it returns nothing —
Letterboxd dates films by festival premiere, TMDB by wider release.


In [8]:
SEARCH_CACHE = Path("data/cache/tmdb_search_raw.json")

def match_all(films, delay=0.05):
    """Match every film to TMDB, caching raw search responses."""
    cache = json.loads(SEARCH_CACHE.read_text()) if SEARCH_CACHE.exists() else {}
    rows, new_calls = [], 0

    for i, film in enumerate(films.itertuples(index=False), start=1):
        key = film.film_key
        year = None if pd.isna(film.film_year) else int(film.film_year)

        if key not in cache:
            results = search_film(film.film_title, year)
            if not results:
                results = search_film(film.film_title)      # year mismatch fallback
            cache[key] = results
            new_calls += 1
            time.sleep(delay)

            if new_calls % 100 == 0:
                SEARCH_CACHE.parent.mkdir(parents=True, exist_ok=True)
                SEARCH_CACHE.write_text(json.dumps(cache))
                print(f"  {i}/{len(films)} processed ({new_calls} API calls)")

        match = best_match(cache[key], film.film_title, year)
        if match is None:
            match = {"tmdb_id": None, "matched_title": None, "matched_year": None,
                     "confidence": "no_match", "similarity": 0.0, "vote_count": None}

        rows.append({"film_key": key, "film_title": film.film_title,
                     "film_year": film.film_year, **match})

    SEARCH_CACHE.parent.mkdir(parents=True, exist_ok=True)
    SEARCH_CACHE.write_text(json.dumps(cache))
    print(f"done — {new_calls} new API calls, {len(rows) - new_calls} from cache")
    return pd.DataFrame(rows)


In [9]:
matches = match_all(films)

done — 0 new API calls, 1176 from cache


## 7. Match report

Decides how much manual work is needed. `no_match` and low tiers are what to watch.


In [10]:
AUTO_ACCEPT = {"exact", "year_off"}

print(f"films: {len(matches)}\n")
print("confidence breakdown")
for tier, n in matches["confidence"].value_counts().items():
    print(f"  {tier:12s} {n:5d}  ({n/len(matches)*100:5.1f}%)")

review = matches[~matches["confidence"].isin(AUTO_ACCEPT)]
print(f"\nauto-accepted: {len(matches) - len(review)} "
      f"({(len(matches)-len(review))/len(matches)*100:.1f}%)")
print(f"needs review : {len(review)}")
print(f"no match at all: {(matches['tmdb_id'].isna()).sum()}")

films: 1176

confidence breakdown
  exact         1097  ( 93.3%)
  year_off        69  (  5.9%)
  weak             8  (  0.7%)
  close            2  (  0.2%)

auto-accepted: 1166 (99.1%)
needs review : 10
no match at all: 0


In [11]:
review[["film_title", "film_year", "matched_title", "matched_year",
        "confidence", "similarity"]].to_string(index=False)

"                          film_title  film_year                                                      matched_title  matched_year confidence  similarity\n                      School of Rock       2003                                                 The School of Rock          2003      close       0.875\n                         Glass Onion       2022                                  Glass Onion: A Knives Out Mystery          2022       weak       0.512\n                       The Evil Dead       1981                                                      The Evil Dead          1983       weak       1.000\n                            Take Out       2004                                                           Take Out          2008       weak       1.000\nMission: Impossible – Dead Reckoning       2023                      Mission: Impossible - Dead Reckoning Part One          2023      close       0.880\n                     World on a Wire       1973 Rainer Werner Fassbinder's World 

### Special cases

Three entries needed intervention, recorded as data rather than hardcoded edits so
the rules survive into other exports.

- **Antiporno (2016)** — TMDB has it as 2017 and flags it adult. Kept as an override
  for documentation; with `include_adult` enabled it should now resolve on its own.
- **World on a Wire (1973)** — Fassbinder's two-part German TV film. Exists on TMDB
  only in the *TV* index (id 86449), so `/search/movie` can never find it.
- **Twin Peaks: The Return (2017)** — television, same reason.

**Pipeline rule:** Letterboxd logs some television as films; those entries have no
TMDB movie record and are excluded.


In [12]:
OVERRIDES = {
    "Antiporno (2016)": 414770,
}

TV_ENTRIES = [
    "World on a Wire (1973)",
    "Twin Peaks: The Return (2017)",
]

for key, tmdb_id in OVERRIDES.items():
    mask = matches["film_key"] == key
    if not mask.any():
        print(f"warning: override key not found — {key!r}")
    matches.loc[mask, "tmdb_id"] = tmdb_id
    matches.loc[mask, "confidence"] = "manual_override"

before = len(matches)
matches = matches[~matches["film_key"].isin(TV_ENTRIES)].copy()

print(f"applied {len(OVERRIDES)} override(s)")
print(f"dropped {before - len(matches)} TV entries -> {len(matches)} films")
print()
print(matches["confidence"].value_counts())

applied 1 override(s)
dropped 2 TV entries -> 1174 films

confidence
exact              1097
year_off             68
weak                  6
close                 2
manual_override       1
Name: count, dtype: int64


### Spot-check the `year_off` tier

69 films auto-accepted on a one-year gap is a lot to take on trust. Every sampled
case showed the same direction — Letterboxd earlier than TMDB — confirming the
premiere-vs-release convention difference rather than mismatches. *Raw*
(Cannes 2016 / release 2017), *The Place Beyond the Pines* (TIFF 2012 / 2013),
*Hit Man* (Venice 2023 / 2024).


In [13]:
matches[matches["confidence"] == "year_off"][
    ["film_title", "film_year", "matched_title", "matched_year"]
].sample(10, random_state=0).to_string(index=False)

'                film_title  film_year              matched_title  matched_year\n                 The Beast       2023                  The Beast          2024\n    The Cabin in the Woods       2011     The Cabin in the Woods          2012\n           The Night House       2020            The Night House          2021\n                       Raw       2016                        Raw          2017\nThe Place Beyond the Pines       2012 The Place Beyond the Pines          2013\n                   Hit Man       2023                    Hit Man          2024\n                Talk to Me       2022                 Talk to Me          2023\n                     Relay       2024                      Relay          2025\n       His Three Daughters       2023        His Three Daughters          2024\n  Welcome to the Dollhouse       1995   Welcome to the Dollhouse          1996'

In [14]:
OUT = Path("data/interim")
OUT.mkdir(parents=True, exist_ok=True)
matches.to_csv(OUT / "tmdb_matches.csv", index=False)
print(f"saved {len(matches)} matched films")

saved 1174 matched films


---

## 8. Enrichment

Search returns `id`, `title`, `overview`, `popularity`, `vote_average`, `vote_count`
and numeric `genre_ids` — but not runtime, named genres, countries, credits or
keywords. Those need a second call per film.

`append_to_response=credits,keywords` folds all three endpoints into one request,
so this is 1,174 calls rather than 3,522.


In [15]:
def fetch_details(tmdb_id):
    r = requests.get(f"https://api.themoviedb.org/3/movie/{tmdb_id}",
                     params={"append_to_response": "credits,keywords"},
                     headers=HEADERS, timeout=15)
    r.raise_for_status()
    return r.json()

d = fetch_details(438631)   # Dune (2021)
print(sorted(d.keys()))

['adult', 'backdrop_path', 'belongs_to_collection', 'budget', 'credits', 'genres', 'homepage', 'id', 'imdb_id', 'keywords', 'origin_country', 'original_language', 'original_title', 'overview', 'popularity', 'poster_path', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'softcore', 'spoken_languages', 'status', 'tagline', 'title', 'video', 'vote_average', 'vote_count']


### Response shapes

Each nested field has a different structure, which determines how it flattens:

- `genres` — list of dicts
- `origin_country` — plain list of ISO codes (simpler than `production_countries`,
  which wraps the same information in dicts)
- `belongs_to_collection` — a dict, or `None` for standalone films
- `keywords` — a dict wrapping a list
- `credits.cast` — ordered by billing, so `order: 0` is the lead
- `credits.crew` — long and unordered (271 entries for *Dune*), needs filtering by job


In [16]:
print("genres              :", d["genres"])
print("origin_country      :", d["origin_country"])
print("production_countries:", d["production_countries"])
print("belongs_to_collection:", d["belongs_to_collection"])
print()
print("keywords keys       :", d["keywords"].keys())
print("keywords sample     :", d["keywords"]["keywords"][:5])
print()
print("credits keys        :", d["credits"].keys())
print("cast[0]             :", d["credits"]["cast"][0])

genres              : [{'id': 878, 'name': 'Science Fiction'}, {'id': 12, 'name': 'Adventure'}]
origin_country      : ['US']
production_countries: [{'iso_3166_1': 'US', 'name': 'United States of America'}]
belongs_to_collection: {'id': 726871, 'name': 'Dune Collection', 'poster_path': '/lxIGYkpvYjLtYtZH684AQft0FhD.jpg', 'backdrop_path': '/fahk0Fu7VUUfK6IkTt1R3waOD9F.jpg'}

keywords keys       : dict_keys(['keywords'])
keywords sample     : [{'id': 11195, 'name': 'empire'}, {'id': 2964, 'name': 'future'}, {'id': 6092, 'name': 'army'}, {'id': 818, 'name': 'based on novel or book'}, {'id': 530, 'name': 'prophecy'}]

credits keys        : dict_keys(['cast', 'crew'])
cast[0]             : {'adult': False, 'gender': 2, 'id': 1190668, 'known_for_department': 'Acting', 'name': 'Timothée Chalamet', 'original_name': 'Timothée Chalamet', 'popularity': 7.1861, 'profile_path': '/dFxpwRpmzpVfP1zjluH68DeQhyj.jpg', 'cast_id': 13, 'character': 'Paul Atreides', 'credit_id': '5b4d01bac3a36823d803cd45', '

In [17]:
crew = d["credits"]["crew"]
print(f"{len(crew)} crew members\n")

from collections import Counter
print("most common jobs:")
for job, n in Counter(c["job"] for c in crew).most_common(10):
    print(f"  {job:28s} {n}")

print()
for c in crew:
    if c["job"] in {"Director", "Director of Photography"}:
        print(f"  {c['job']:26s} {c['name']}")

271 crew members

most common jobs:
  Stunts                       75
  Executive Producer           10
  Stunt Double                 9
  Prosthetic Makeup Artist     7
  Visual Effects               6
  Sound Effects Editor         5
  ADR Mixer                    5
  Art Direction                5
  Concept Artist               5
  Makeup Artist                4

  Director                   Denis Villeneuve
  Director of Photography    Greig Fraser


### Parsing

Two choices worth noting.

**`.get()` throughout rather than `[...]`** — across 1,174 films some will lack a
runtime, keywords or crew entirely, and one sparse film should not kill the run.

**Lists joined with `|` rather than kept as lists** — this goes to CSV, and Python
lists round-trip as the string `"['Drama', 'Thriller']"`. Pipes split cleanly and
don't occur in genre or person names the way commas do.

Crew is filtered by exact job string. `first_with_job` takes the first match rather
than assuming exactly one, since co-directed films and multiple DPs exist.


In [18]:
def parse_details(d):
    """Flatten a TMDB detail response into a single flat record."""
    crew = d.get("credits", {}).get("crew", [])
    cast = d.get("credits", {}).get("cast", [])

    def first_with_job(job):
        for c in crew:
            if c.get("job") == job:
                return c.get("name")
        return None

    collection = d.get("belongs_to_collection")

    return {
        "tmdb_id":         d.get("id"),
        "tmdb_title":      d.get("title"),
        "release_date":    d.get("release_date"),
        "runtime":         d.get("runtime"),
        "original_language": d.get("original_language"),
        "origin_country":  "|".join(d.get("origin_country") or []),
        "genres":          "|".join(g["name"] for g in d.get("genres", [])),
        "keywords":        "|".join(k["name"] for k in d.get("keywords", {}).get("keywords", [])),
        "overview":        d.get("overview"),
        "vote_average":    d.get("vote_average"),
        "vote_count":      d.get("vote_count"),
        "popularity":      d.get("popularity"),
        "in_collection":   collection is not None,
        "collection_name": collection["name"] if collection else None,
        "director":        first_with_job("Director"),
        "cinematographer": first_with_job("Director of Photography"),
        "cast_top5":       "|".join(c["name"] for c in cast[:5]),
    }

parse_details(d)

{'tmdb_id': 438631,
 'tmdb_title': 'Dune',
 'release_date': '2021-09-15',
 'runtime': 155,
 'original_language': 'en',
 'origin_country': 'US',
 'genres': 'Science Fiction|Adventure',
 'keywords': 'empire|future|army|based on novel or book|prophecy|dystopia|emperor|sand|spice|chosen one|hallucinogen|treason|baron|revenge|premonition|betrayal|space|water shortage|creature|desert|knife fight|destiny|giant worm|space opera|sand dune|messiah|complex|mother son relationship|giant creature',
 'overview': "Paul Atreides, a brilliant and gifted young man born into a great destiny beyond his understanding, must travel to the most dangerous planet in the universe to ensure the future of his family and his people. As malevolent forces explode into conflict over the planet's exclusive supply of the most precious resource in existence - a commodity capable of unlocking humanity's greatest potential - only those who can conquer their fear will survive.",
 'vote_average': 7.785,
 'vote_count': 15560,

### Fetch all details

Unlike the search cache, this stores the **raw** response and parses on the way out —
so adding a field to `parse_details` later costs no API calls.

JSON object keys are always strings, hence `str(int(tmdb_id))` to avoid a
write-int/read-string mismatch.


In [19]:
DETAILS_CACHE = Path("data/cache/tmdb_details.json")

def fetch_all_details(tmdb_ids, delay=0.05):
    """Fetch and parse detail records for every TMDB ID, caching raw responses."""
    cache = json.loads(DETAILS_CACHE.read_text()) if DETAILS_CACHE.exists() else {}
    rows, new_calls = [], 0

    for i, tmdb_id in enumerate(tmdb_ids, start=1):
        key = str(int(tmdb_id))

        if key not in cache:
            cache[key] = fetch_details(int(tmdb_id))
            new_calls += 1
            time.sleep(delay)

            if new_calls % 100 == 0:
                DETAILS_CACHE.parent.mkdir(parents=True, exist_ok=True)
                DETAILS_CACHE.write_text(json.dumps(cache))
                print(f"  {i}/{len(tmdb_ids)} processed ({new_calls} API calls)")

        rows.append(parse_details(cache[key]))

    DETAILS_CACHE.parent.mkdir(parents=True, exist_ok=True)
    DETAILS_CACHE.write_text(json.dumps(cache))
    print(f"done — {new_calls} new API calls, {len(rows) - new_calls} from cache")
    return pd.DataFrame(rows)

In [20]:
details = fetch_all_details(matches["tmdb_id"])
print(details.shape)

done — 0 new API calls, 1174 from cache
(1174, 17)


## 9. Validation

Sparse fields are what bite later. Runtime is checked for `0` as well as null,
because TMDB uses zero as an unknown placeholder and a film with runtime 0 would
quietly poison any feature built on it.

`collection_name` nulls are **not** missing data — they are films that belong to no
franchise. `in_collection` carries that correctly as a boolean.


In [21]:
print("missing values:")
print(details.isna().sum()[lambda s: s > 0])
print()
print(f"runtime = 0 or null : {((details['runtime'] == 0) | details['runtime'].isna()).sum()}")
print(f"no genres           : {(details['genres'] == '').sum()}")
print(f"no keywords         : {(details['keywords'] == '').sum()}")
print(f"no director         : {details['director'].isna().sum()}")
print(f"no cinematographer  : {details['cinematographer'].isna().sum()}")
print(f"zero vote_count     : {(details['vote_count'] == 0).sum()}")

missing values:
collection_name    968
cinematographer     15
dtype: int64

runtime = 0 or null : 0
no genres           : 0
no keywords         : 7
no director         : 0
no cinematographer  : 15
zero vote_count     : 0


In [22]:
details[details["director"].isna() | (details["genres"] == "") | (details["vote_count"] == 0)][
    ["tmdb_id", "tmdb_title", "release_date", "runtime", "genres", "director", "vote_count"]
]

,tmdb_id,tmdb_title,release_date,runtime,genres,director,vote_count


### Merge and check for silent suffixing

Both frames carry `vote_count` — `matches` picked one up from the search results as a
tiebreaker, but the detail response is authoritative, so the search copy is dropped
before merging. Without that, pandas silently produces `vote_count_x` and
`vote_count_y` and you end up with two subtly different copies and no idea which one
the model used.


In [23]:
enriched = matches.drop(columns=["vote_count"]).merge(details, on="tmdb_id", how="left")

print(enriched.shape)
print([c for c in enriched.columns if c.endswith(("_x", "_y"))] or "no suffixed columns")

(1174, 24)
no suffixed columns


### Hunt for title-collision mismatches

Low vote count doesn't mean wrong — genuinely obscure films exist in this history —
but it is where wrong matches concentrate. The tell is a **runtime inconsistent with
the film in question**: *Under the Skin* at 19 minutes and *Talk to Me* at 6 minutes
were both fragments sharing a title with a feature.

These four cases are what motivated the demotion rule in §5. Re-run after that fix,
this list should contain only genuinely obscure films.


In [24]:
suspicious = enriched[(enriched["vote_count"] < 50) & (enriched["confidence"] == "exact")]
print(f"{len(suspicious)} exact matches with under 50 votes\n")
print(suspicious[["film_title", "film_year", "tmdb_title", "release_date",
                  "vote_count", "runtime"]].to_string(index=False))

9 exact matches with under 50 votes

         film_title  film_year          tmdb_title release_date  vote_count  runtime
                 36       2012                  36   2012-01-17          22       68
From What Is Before       2014 From What Is Before   2014-07-03          37      345
     Apart from You       1933      Apart from You   1933-04-01          23       64
     Le Grand Amour       1969      Le Grand Amour   1969-03-21          29       87
            Mahjong       1996             Mahjong   1996-12-07          49      121
               Coma       2022                Coma   2022-11-16          31       81
           Magellan       2025            Magellan   2025-09-10          37      164
 Beijing Watermelon       1989  Beijing Watermelon   1989-11-18           7      135
    Winter Brothers       2017     Winter Brothers   2017-12-07          42       93


## 10. Save

In [25]:
OUT = Path("data/interim")
OUT.mkdir(parents=True, exist_ok=True)

enriched.to_csv(OUT / "films_enriched.csv", index=False)
print(f"saved {len(enriched)} films, {enriched.shape[1]} columns -> {OUT / 'films_enriched.csv'}")
print(list(enriched.columns))


saved 1174 films, 24 columns -> data/interim/films_enriched.csv
['film_key', 'film_title', 'film_year', 'tmdb_id', 'matched_title', 'matched_year', 'confidence', 'similarity', 'tmdb_title', 'release_date', 'runtime', 'original_language', 'origin_country', 'genres', 'keywords', 'overview', 'vote_average', 'vote_count', 'popularity', 'in_collection', 'collection_name', 'director', 'cinematographer', 'cast_top5']


---

## 11. Other viewers' reviews

**Why this exists.** An unseen film has no review from this viewer, so review-derived
information can only reach it through something the film already has. An earlier attempt
bridged reviews to TMDB *overviews* — but an overview is an objective description and a
review is a subjective judgement, so matching one against the other assumes a
relationship that isn't there.

The justifiable bridge is **like-for-like: subjective text to subjective text.** The
unseen film may have reviews from *other people*. The idea was to apply the viewer's own
learned word weights (R1, in `04`) to those reviews — so that when other reviewers call a
film "interesting" or "decent", the viewer's personal reading of those words (both
predict *low* ratings for this viewer) is what gets applied.

This section only acquires the data and tests whether coverage is sufficient.


In [26]:
def fetch_reviews(tmdb_id, page=1):
    r = requests.get(f"https://api.themoviedb.org/3/movie/{tmdb_id}/reviews",
                     params={"page": page}, headers=HEADERS, timeout=15)
    r.raise_for_status()
    return r.json()

d = fetch_reviews(438631)   # Dune (2021)
print(f"total reviews: {d['total_results']}   pages: {d['total_pages']}")
print(sorted(d["results"][0].keys()))
print()
print(d["results"][0]["content"][:400])

total reviews: 17   pages: 1
['author', 'author_details', 'content', 'created_at', 'id', 'updated_at', 'url']


**FABULOUS 🥇🥇🥇🥇 . . . . And , Oh , Yes . . . . Hans Zimmer's Score's Already Got "OSCAR" Written On It 😉 ; & EXPECT A WHOLE " HOST OF OTHER _MAJOR_ NOMINATIONS - AS WELL "**

This Is A **- _B I G_ -** Screen - MINI - Review. Picture Viewed Oct. 07, 2021 ; At Vox Cinemas , U . A . E

______________________________________________________

Paul Atreides : " Fear is the mind-killer. Fear is t


*Dune* has only **17** reviews — one of the most-watched films of the decade — which
is an early warning that obscure films will often have none.

The text also needs different cleaning from Letterboxd reviews: markdown rather than
HTML, emoji, shouted title case, and **quoted dialogue** ("Fear is the mind-killer"),
which is not the reviewer's opinion at all.

**`created_at` is the useful discovery.** Every review is timestamped, so a feature can
use only reviews written *before* the viewer watched the film — genuinely historical,
unlike `vote_average`, which can only be fetched at its current value.


### Coverage

Page 1 only, which carries `total_results` and up to 20 reviews — enough for nearly every
film. Raw responses cached as with the other fetches.

**Viability threshold, set before running:** at least half of films need three or more
reviews. One or two reviews give a very noisy score dominated by a single reviewer.


In [27]:
REVIEWS_CACHE = Path("data/cache/tmdb_reviews.json")

def fetch_all_reviews(tmdb_ids, delay=0.05):
    """Page 1 of TMDB reviews per film, raw responses cached by ID."""
    cache = json.loads(REVIEWS_CACHE.read_text()) if REVIEWS_CACHE.exists() else {}
    new_calls = 0

    for i, tmdb_id in enumerate(tmdb_ids, start=1):
        key = str(int(tmdb_id))
        if key not in cache:
            cache[key] = fetch_reviews(int(tmdb_id))
            new_calls += 1
            time.sleep(delay)
            if new_calls % 100 == 0:
                REVIEWS_CACHE.parent.mkdir(parents=True, exist_ok=True)
                REVIEWS_CACHE.write_text(json.dumps(cache))
                print(f"  {i}/{len(tmdb_ids)} processed ({new_calls} API calls)")

    REVIEWS_CACHE.parent.mkdir(parents=True, exist_ok=True)
    REVIEWS_CACHE.write_text(json.dumps(cache))
    print(f"done — {new_calls} new API calls")
    return cache

reviews_raw = fetch_all_reviews(enriched["tmdb_id"])

counts = pd.Series({k: v["total_results"] for k, v in reviews_raw.items()})
print()
print(f"films with 0 reviews : {(counts == 0).sum()} ({(counts == 0).mean():.1%})")
print(f"films with 1–2       : {counts.between(1, 2).sum()}")
print(f"films with 3+        : {(counts >= 3).sum()} ({(counts >= 3).mean():.1%})")
print(f"median reviews/film  : {counts.median():.0f}")

done — 0 new API calls

films with 0 reviews : 164 (14.0%)
films with 1–2       : 363
films with 3+        : 647 (55.1%)
median reviews/film  : 3


55.1% with three or more — but these are **total** reviews, written at any time up to
now. The feature may only use reviews written *before each viewing*. For a film watched
near release, most reviews postdate the viewing.

Counted **per viewing** rather than per film, because the cutoff date differs per
viewing — a rewatch has more prior reviews than the first watch.

`utc=True` then `tz_convert(None)`: TMDB timestamps carry a timezone and watched dates
don't, and comparing the two directly raises.


In [28]:
view = viewings.merge(enriched[["film_key", "tmdb_id"]], on="film_key", how="inner")
view["watched_date"] = pd.to_datetime(view["watched_date"])
view = view.sort_values("watched_date").reset_index(drop=True)

def n_prior_reviews(tmdb_id, watched):
    res = reviews_raw.get(str(int(tmdb_id)), {}).get("results", [])
    if not res:
        return 0
    created = pd.to_datetime([r["created_at"] for r in res], utc=True).tz_convert(None)
    return int((created < watched).sum())

view["n_prior_reviews"] = [n_prior_reviews(t, w) for t, w in zip(view["tmdb_id"], view["watched_date"])]

split = int(len(view) * 0.8)
for name, part in [("all", view), ("train", view[:split]), ("test", view[split:])]:
    n = part["n_prior_reviews"]
    print(f"{name:5s}  n={len(part):4d}   0 reviews: {(n == 0).mean():5.1%}   3+: {(n >= 3).mean():5.1%}")

all    n=1192   0 reviews: 23.2%   3+: 32.3%
train  n= 953   0 reviews: 23.2%   3+: 32.3%
test   n= 239   0 reviews: 23.0%   3+: 32.2%


**Below the threshold.** Only 32% of viewings have three or more prior reviews;
23% have none. Train and test are nearly identical (32.3% vs 32.2%) — the test period was
expected to be worse, since recent viewing skews toward new releases, and it wasn't.

The arithmetic makes this more than a rule of thumb. At n=239 the smallest detectable
improvement is roughly 0.035 MAE. With usable signal on only a third of films, the feature
would need to cut error on *those* films by ~0.11 stars each — far beyond what R1 managed
even reading each film's *own* review.

**Conclusion:** the idea is sound, the data is too sparse before the viewer's watch dates.
Not built. Recorded as further work — Letterboxd holds far more reviews than TMDB, but has
no public API.

*Known gap:* only page 1 is cached, so films with more than 20 reviews lose a few. Not
worth fetching further pages given the coverage result.


---

## Summary

| | |
|---|---|
| Films matched | 1,174 (from 1,176; 2 TV entries dropped) |
| Auto-accepted (`exact` / `year_off`) | ~99% |
| Total match failures | 0 |
| Cinematographer coverage | 98.7% |
| Runtime coverage | 100% |
| Viewings with 3+ prior TMDB reviews | 32% (below the 50% viability threshold) |

**Known limitations**

- TMDB fields are fetched *today*, not as they stood on the watch date.
  `vote_average`, `vote_count` and `popularity` therefore postdate viewing — an
  unavoidable approximation, not a clean temporal cut.
- Where title and year cannot separate two candidates, the better-known film wins.
  Defensible, but wrong exactly when a film is genuinely obscure.
- Letterboxd TV entries have no movie record and are excluded.
- TMDB user reviews are too sparse before each watch date to support a feature.

**Next:** `03_eda.ipynb` joins this to `viewings.csv` for exploratory analysis.
